# Figure Creation

In [1]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, precision_recall_curve
from sklearn.calibration import calibration_curve
import os

# =============================================================================
# SCRIPT CONFIGURATION
# =============================================================================
# --- Analysis Mode ---
# Set to True to run for all sources in ALL_DATA_SOURCES.
# Set to False to run only for the single source specified in DATA_SOURCE.
ANALYZE_ALL_SOURCES = True

# --- NEW: Plotting Options ---
# Set to True to display shaded confidence intervals on ROC and PR curves.
PLOT_CONFIDENCE_INTERVALS = True

# --- Select which dataset's results you want to analyze (if ANALYZE_ALL_SOURCES is False) ---
DATA_SOURCE = "intraop"

# --- List of all possible data sources (used when ANALYZE_ALL_SOURCES is True) ---
ALL_DATA_SOURCES = ["preop", "intraop", "combined"]

# --- Paths and Naming ---
RESULTS_DIR = '/home/server/Projects/data/AKI/results/'
FIGURES_DIR = 'figures'  # Define the folder to save plots

# Note: The full path will be constructed dynamically in the analysis function.

# --- Model Name Translations for Plots ---
model_translate = {
    'base': 'base', 'base_54k': 'base_54k', # Add any new base models here
    'log_reg': 'Logistic Regression', 'xgb': 'GBT',
    'svm_linear': 'SVM (Linear)', 'mlp': 'MLP', 'rf': 'Random Forest',
    'knn': 'KNN', 'asa_rule': 'ASA Rule', 'autogluon': 'AutoGluon',
    'lstm': 'LSTM', 'hybrid': 'Hybrid (MLP + LSTM)'
}

# Define which ground-truth 'base' model to use for specific predictive models.
# This allows comparing models trained on different underlying datasets.
# Models not listed here will use the DEFAULT_BASE.
model_base_map = {
    'LSTM': 'base_54k',
    'Hybrid (MLP + LSTM)': 'base_54k'
}
DEFAULT_BASE = 'base'


# --- Consistent Color Mapping for Models ---
model_colors = {
    'Logistic Regression': 'green',
    'GBT': 'red',
    'SVM (Linear)': 'purple',
    'MLP': 'brown',
    'Random Forest': 'crimson',
    'KNN': 'gray',
    'ASA Rule': 'orange',
    'AutoGluon': 'blue',
    'LSTM': 'darkorange',
    'Hybrid (MLP + LSTM)': 'teal'
}


# =============================================================================
# PLOTTING FUNCTIONS (ADAPTED FOR NEW .pkl FORMAT)
# =============================================================================

def plot_roc_curves(df, data_source_name, save_dir):
    """Plots the mean ROC curve for each model, optionally with a shaded confidence interval."""
    plt.figure(figsize=(10, 8), dpi=300)
    
    # Load all available 'base' models' ground-truth data
    base_names = [name for name in df['model_name'].unique() if 'base' in name]
    if not base_names:
        print("ERROR: No 'base' model with true labels found. Cannot plot curves.")
        return
        
    base_models_data = {
        name: df[df['model_name'] == name]['y_pred_binary'].values[0]
        for name in base_names
    }
    
    df_to_plot = df[~df['model_name'].str.contains('base')]
    
    # Pre-calculate AUCs to sort the legend
    model_performance = []
    for index, model_row in df_to_plot.iterrows():
        model_name = model_row['model_name']
        
        # Determine which base model's ground truth to use
        base_to_use = model_base_map.get(model_name, DEFAULT_BASE)
        if base_to_use not in base_models_data:
            print(f"WARNING [ROC]: Base model '{base_to_use}' for model '{model_name}' not found. Skipping.")
            continue
        y_true_runs = base_models_data[base_to_use]

        aucs = []
        for i in range(len(model_row['y_prob'])):
            if i >= len(y_true_runs) or len(y_true_runs[i]) != len(model_row['y_prob'][i]):
                print(f"WARNING [ROC]: Skipping fold {i} for model '{model_name}'. Inconsistent sample numbers.")
                continue
            fpr, tpr, _ = roc_curve(y_true_runs[i], model_row['y_prob'][i])
            aucs.append(auc(fpr, tpr))

        if not aucs:
            print(f"WARNING [ROC]: Could not calculate AUC for model '{model_name}'. Skipping.")
            continue
            
        model_performance.append((model_name, np.mean(aucs), model_row))
    
    model_performance.sort(key=lambda x: x[1], reverse=True)

    for model_name, mean_auc, model_row in model_performance:
        base_to_use = model_base_map.get(model_name, DEFAULT_BASE)
        y_true_runs = base_models_data[base_to_use]
        y_prob_runs = model_row['y_prob']
        
        mean_fpr = np.linspace(0, 1, 100)
        tprs = []
        for i in range(len(y_prob_runs)):
            if i >= len(y_true_runs) or len(y_true_runs[i]) != len(y_prob_runs[i]):
                continue
            fpr, tpr, _ = roc_curve(y_true_runs[i], y_prob_runs[i])
            interp_tpr = np.interp(mean_fpr, fpr, tpr)
            interp_tpr[0] = 0.0
            tprs.append(interp_tpr)
        
        if not tprs: continue
            
        mean_tpr = np.mean(tprs, axis=0)
        mean_tpr[-1] = 1.0
        color = model_colors.get(model_name, None)
        
        # --- MODIFIED: Conditionally plot confidence interval ---
        if PLOT_CONFIDENCE_INTERVALS:
            # The shaded region represents +/- 1 standard deviation from the mean TPR.
            std_tpr = np.std(tprs, axis=0)
            tprs_upper = np.minimum(mean_tpr + std_tpr, 1)
            tprs_lower = np.maximum(mean_tpr - std_tpr, 0)
            plt.fill_between(mean_fpr, tprs_lower, tprs_upper, color=color, alpha=0.2)
        # --- END MODIFIED ---

        valid_aucs = [auc(roc_curve(y_true_runs[i], y_prob_runs[i])[0], roc_curve(y_true_runs[i], y_prob_runs[i])[1]) for i in range(len(y_prob_runs)) if i < len(y_true_runs) and len(y_true_runs[i]) == len(y_prob_runs[i])]
        std_auc = np.std(valid_aucs)
        
        plt.plot(mean_fpr, mean_tpr, color=color, label=f'{model_name} (AUC = {mean_auc:.2f} \u00B1 {std_auc:.2f})', lw=2)

    plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Chance')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'Mean ROC Curves on {data_source_name.upper()} Data')
    plt.legend(loc='lower right', fontsize='large')
    plt.grid(True)
    
    # Save the figure
    save_path = os.path.join(save_dir, f'roc_curves_{data_source_name}.png')
    plt.savefig(save_path, bbox_inches='tight')
    print(f"Saved ROC curve plot to: {save_path}")
    plt.close() # Close the plot to free memory

def plot_pr_curves(df, data_source_name, save_dir):
    """Plots the mean Precision-Recall curve for each model, optionally with a shaded confidence interval."""
    plt.figure(figsize=(10, 8), dpi=300)
    
    base_names = [name for name in df['model_name'].unique() if 'base' in name]
    if not base_names:
        print("ERROR: No 'base' model with true labels found. Cannot plot curves.")
        return
        
    base_models_data = {
        name: df[df['model_name'] == name]['y_pred_binary'].values[0]
        for name in base_names
    }
    
    df_to_plot = df[~df['model_name'].str.contains('base')]
    
    model_performance = []
    for index, model_row in df_to_plot.iterrows():
        model_name = model_row['model_name']
        base_to_use = model_base_map.get(model_name, DEFAULT_BASE)
        if base_to_use not in base_models_data:
            print(f"WARNING [PR]: Base model '{base_to_use}' for model '{model_name}' not found. Skipping.")
            continue
        y_true_runs = base_models_data[base_to_use]
        
        pr_aucs = []
        for i in range(len(model_row['y_prob'])):
            if i >= len(y_true_runs) or len(y_true_runs[i]) != len(model_row['y_prob'][i]):
                print(f"WARNING [PR]: Skipping fold {i} for model '{model_name}'. Inconsistent sample numbers.")
                continue
            precision, recall, _ = precision_recall_curve(y_true_runs[i], model_row['y_prob'][i])
            pr_aucs.append(auc(recall, precision))

        if not pr_aucs:
            print(f"WARNING [PR]: Could not calculate AUC for model '{model_name}'. Skipping.")
            continue
            
        model_performance.append((model_row['model_name'], np.mean(pr_aucs), model_row))

    model_performance.sort(key=lambda x: x[1], reverse=True)

    for model_name, mean_pr_auc, model_row in model_performance:
        base_to_use = model_base_map.get(model_name, DEFAULT_BASE)
        y_true_runs = base_models_data[base_to_use]
        y_prob_runs = model_row['y_prob']
        
        mean_recall = np.linspace(0, 1, 100)
        precisions = []
        for i in range(len(y_prob_runs)):
            if i >= len(y_true_runs) or len(y_true_runs[i]) != len(y_prob_runs[i]):
                continue
            precision, recall, _ = precision_recall_curve(y_true_runs[i], y_prob_runs[i])
            # Interpolate precision values for a common recall axis
            interp_prec = np.interp(mean_recall, recall[::-1], precision[::-1])
            precisions.append(interp_prec)

        if not precisions: continue

        mean_prec = np.mean(precisions, axis=0)
        color = model_colors.get(model_name, None)
        
        # --- MODIFIED: Conditionally plot confidence interval ---
        if PLOT_CONFIDENCE_INTERVALS:
            # The shaded region represents +/- 1 standard deviation from the mean precision.
            std_prec = np.std(precisions, axis=0)
            prec_upper = np.minimum(mean_prec + std_prec, 1)
            prec_lower = np.maximum(mean_prec - std_prec, 0)
            plt.fill_between(mean_recall, prec_lower, prec_upper, color=color, alpha=0.2)
        # --- END MODIFIED ---

        valid_pr_aucs = [auc(precision_recall_curve(y_true_runs[i], y_prob_runs[i])[1], precision_recall_curve(y_true_runs[i], y_prob_runs[i])[0]) for i in range(len(y_prob_runs)) if i < len(y_true_runs) and len(y_true_runs[i]) == len(y_prob_runs[i])]
        std_pr_auc = np.std(valid_pr_aucs)

        plt.plot(mean_recall, mean_prec, color=color, label=f'{model_name} (AUC = {mean_pr_auc:.2f} \u00B1 {std_pr_auc:.2f})', lw=2)

    # Use the default base for the 'no skill' line as a general reference
    y_true_default_runs = base_models_data[DEFAULT_BASE]
    y_true_default_flat = np.concatenate(y_true_default_runs)
    no_skill = len(y_true_default_flat[y_true_default_flat==1]) / len(y_true_default_flat)
    plt.plot([0, 1], [no_skill, no_skill], linestyle='--', color='gray', label='No Skill')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title(f'Mean Precision-Recall Curves on {data_source_name.upper()} Data')
    plt.legend(loc='upper right', fontsize='large')
    plt.grid(True)
    
    # Save the figure
    save_path = os.path.join(save_dir, f'pr_curves_{data_source_name}.png')
    plt.savefig(save_path, bbox_inches='tight')
    print(f"Saved Precision-Recall curve plot to: {save_path}")
    plt.close() # Close the plot to free memory

def plot_calibration_curves(df, data_source_name, save_dir):
    """Plots calibration curves by pooling all predictions and saves the figure."""
    plt.figure(figsize=(10, 8), dpi=300)
    
    base_names = [name for name in df['model_name'].unique() if 'base' in name]
    if not base_names:
        print("ERROR: No 'base' model with true labels found. Cannot plot curves.")
        return
        
    base_models_data = {
        name: df[df['model_name'] == name]['y_pred_binary'].values[0]
        for name in base_names
    }

    df_to_plot = df[~df['model_name'].str.contains('base')]

    for index, model_row in df_to_plot.iterrows():
        model_name = model_row['model_name']
        base_to_use = model_base_map.get(model_name, DEFAULT_BASE)
        if base_to_use not in base_models_data:
            print(f"WARNING [Calibration]: Base model '{base_to_use}' for model '{model_name}' not found. Skipping.")
            continue
            
        y_true_runs = base_models_data[base_to_use]
        y_prob_runs = model_row['y_prob']
        
        # Check for total length consistency before concatenating
        len_y_true = sum(len(arr) for arr in y_true_runs)
        len_y_prob = sum(len(arr) for arr in y_prob_runs)

        if len_y_true != len_y_prob:
            print(f"WARNING [Calibration]: Skipping model '{model_name}'. Pooled predictions have mismatched samples.")
            continue
            
        y_true_all = np.concatenate(y_true_runs)
        y_prob_all = np.concatenate(y_prob_runs)

        prob_true, prob_pred = calibration_curve(y_true_all, y_prob_all, n_bins=15, strategy='uniform')
        plt.plot(prob_pred, prob_true, marker='o', linestyle='-', color=model_colors.get(model_name, None), label=f'{model_name}')

    plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Perfectly Calibrated')
    plt.xlabel('Mean Predicted Probability')
    plt.ylabel('Fraction of Positives')
    plt.title(f'Calibration Curves on {data_source_name.upper()} Data')
    plt.legend(loc='upper left', fontsize='large')
    plt.grid(True)
    
    # Save the figure
    save_path = os.path.join(save_dir, f'calibration_curves_{data_source_name}.png')
    plt.savefig(save_path, bbox_inches='tight')
    print(f"Saved calibration curve plot to: {save_path}")
    plt.close() # Close the plot to free memory

# =============================================================================
# METRICS TABLE FUNCTION
# =============================================================================
def print_metrics_table(df, data_source_name):
    """Prints a formatted table of performance metrics with 95% CIs."""
    df_to_print = df[~df['model_name'].str.contains('base')].copy()
    
    metrics_to_print = ['roc_auc', 'balanced_accuracy', 'recall', 'precision', 'specificity', 'f1']
    
    print("\n" + "="*80)
    print(f"PERFORMANCE METRICS SUMMARY ({data_source_name.upper()})")
    print("="*80)
    print(f"{'Model':<25} {'Metric':<20} {'Mean':<10} {'95% CI Lower':<15} {'95% CI Upper':<15}")
    print("-"*80)

    for index, model_row in df_to_print.iterrows():
        model_name = model_row['model_name']
        print(f"{model_name:<25}")
        for metric in metrics_to_print:
            if metric not in model_row or model_row[metric] is None or len(np.atleast_1d(model_row[metric])) == 0:
                print(f"{'':<25} {metric:<20} {'N/A':<10} {'N/A':<15} {'N/A':<15}")
                continue

            arr = model_row[metric]
            
            if np.isscalar(arr) or len(arr) == 1:
                mean = np.mean(arr)
                ci_lower, ci_upper = "N/A", "N/A"
            else:
                mean = np.mean(arr)
                sem = stats.sem(arr)
                n = len(arr)

                if sem > 0 and n > 1:
                    ci = stats.t.interval(0.95, df=n-1, loc=mean, scale=sem)
                    ci_lower, ci_upper = f"{ci[0]:.4f}", f"{ci[1]:.4f}"
                else:
                    ci_lower, ci_upper = "N/A", "N/A"
            
            print(f"{'':<25} {metric:<20} {mean:<10.4f} {ci_lower:<15} {ci_upper:<15}")
        print("-"*80)

# =============================================================================
# MAIN ANALYSIS EXECUTION
# =============================================================================
def run_full_analysis(data_source):
    """
    Loads data for a given source and runs all plotting and metric functions.
    """
    print(f"\n{'~'*25} RUNNING ANALYSIS FOR: {data_source.upper()} {'~'*25}\n")
    
    # Create the figures directory if it doesn't exist
    os.makedirs(FIGURES_DIR, exist_ok=True)
    
    output_pkl = os.path.join(RESULTS_DIR, f"tabular_{data_source}_test.pkl")

    try:
        df = pd.read_pickle(output_pkl)
        df['model_name'] = df['model_name'].map(model_translate)
        df.dropna(subset=['model_name'], inplace=True)
        print(f"Successfully loaded and processed results from: {output_pkl}")
        found_models = df[~df['model_name'].str.contains('base')]['model_name'].unique().tolist()
        print(f"Found models: {found_models}")
    except FileNotFoundError:
        print(f"ERROR: Results file not found at {output_pkl}. Skipping this data source.")
        return

    # Pass the save directory to the plotting functions
    plot_roc_curves(df, data_source, FIGURES_DIR)
    plot_pr_curves(df, data_source, FIGURES_DIR)
    plot_calibration_curves(df, data_source, FIGURES_DIR)
    
    print_metrics_table(df, data_source)
    print(f"\n{'~'*25} COMPLETED ANALYSIS FOR: {data_source.upper()} {'~'*25}\n")


if __name__ == "__main__":
    if ANALYZE_ALL_SOURCES:
        for source in ALL_DATA_SOURCES:
            run_full_analysis(source)
    else:
        run_full_analysis(DATA_SOURCE)



~~~~~~~~~~~~~~~~~~~~~~~~~ RUNNING ANALYSIS FOR: PREOP ~~~~~~~~~~~~~~~~~~~~~~~~~

Successfully loaded and processed results from: /home/server/Projects/data/AKI/results/tabular_preop_test.pkl
Found models: ['Logistic Regression', 'AutoGluon', 'GBT', 'Random Forest', 'MLP', 'KNN', 'ASA Rule']
Saved ROC curve plot to: figures/roc_curves_preop.png
Saved Precision-Recall curve plot to: figures/pr_curves_preop.png
Saved calibration curve plot to: figures/calibration_curves_preop.png

PERFORMANCE METRICS SUMMARY (PREOP)
Model                     Metric               Mean       95% CI Lower    95% CI Upper   
--------------------------------------------------------------------------------
Logistic Regression      
                          roc_auc              0.9122     0.9097          0.9148         
                          balanced_accuracy    0.8410     0.8380          0.8439         
                          recall               0.8043     0.7982          0.8105         
             

# Performance Metrics Table

In [2]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import os
from sklearn.metrics import precision_recall_curve, auc
import html
import uuid

# =============================================================================
# SCRIPT CONFIGURATION
# =============================================================================
# --- List of data sources to include in the table ---
DATA_SOURCES = ["preop", "intraop", "combined"]

# --- File Paths ---
RESULTS_DIR = '/home/server/Projects/data/AKI/results/'
OUTPUT_FILE = 'performance_table.md'

# --- Model Name Translations for Plots ---
# This ensures consistency with your main analysis script
model_translate = {
    'base': 'base', 'base_54k': 'base_54k',
    'log_reg': 'LR', 'xgb': 'GBT',
    'svm_linear': 'SVM', 'mlp': 'MLP', 'rf': 'RF',
    'knn': 'KNN', 'asa_rule': 'ASA Rule', 'autogluon': 'AutoGluon',
    'lstm': 'LSTM', 'hybrid': 'MLP+LSTM'
}

# --- Base Model Mapping ---
# Defines which ground-truth 'base' model to use for specific predictive models.
model_base_map = {
    'LSTM': 'base_54k',
    'MLP+LSTM': 'base_54k'
}
DEFAULT_BASE = 'base'


# --- Metrics to include in the table and their display names ---
# The order in this dictionary determines the column order in the table
METRICS_MAP = {
    'roc_auc': 'AUROC',
    'pr_auc': 'AUPRC',
    'recall': 'Sensitivity',
    'specificity': 'Specificity',
    'precision': 'Precision',
    'f1': 'F-score',
    'balanced_accuracy': 'Accuracy' # Using balanced accuracy as per your example
}

# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def get_metric_ci(metric_array):
    """Calculates the mean and 95% CI for a given array of metric scores."""
    if metric_array is None or pd.isnull(metric_array).all() or len(np.atleast_1d(metric_array)) == 0:
        return "N/A", "N/A"
        
    arr = np.atleast_1d(metric_array)
    
    if np.isscalar(arr) or len(arr) == 1:
        mean_str = f"{np.mean(arr):.3f}"
        ci_str = "N/A"
    else:
        mean = np.mean(arr)
        sem = stats.sem(arr)
        n = len(arr)
        mean_str = f"{mean:.3f}"

        if sem > 0 and n > 1:
            ci = stats.t.interval(0.95, df=n-1, loc=mean, scale=sem)
            ci_str = f"({ci[0]:.3f}, {ci[1]:.3f})"
        else:
            ci_str = "N/A"
            
    return mean_str, ci_str

# =============================================================================
# MAIN SCRIPT LOGIC
# =============================================================================

def generate_performance_tables():
    """
    Generates and displays performance tables in both Markdown and rich-text HTML format.
    """
    all_results = {}
    base_models_data = {}

    # --- 1. Load data from all specified sources ---
    for source in DATA_SOURCES:
        pkl_path = os.path.join(RESULTS_DIR, f"tabular_{source}_test.pkl")
        try:
            df = pd.read_pickle(pkl_path)
            df['model_name'] = df['model_name'].map(model_translate)
            df.dropna(subset=['model_name'], inplace=True)
            
            df_models = df[~df['model_name'].str.contains('base', na=False)].copy()
            df_base = df[df['model_name'].str.contains('base', na=False)].copy()
            
            all_results[source] = df_models
            
            for _, base_row in df_base.iterrows():
                base_models_data[base_row['model_name']] = base_row['y_pred_binary']
                
            print(f"Successfully loaded data for: {source.upper()}")
        except FileNotFoundError:
            print(f"WARNING: Results file not found for '{source}'. Skipping.")
            continue

    # --- 2. Initialize Markdown and HTML table strings ---
    markdown_lines = []
    md_header = "| Model | " + " | ".join(METRICS_MAP.values()) + " |"
    md_separator = "|" + "---|" * (len(METRICS_MAP) + 1)
    markdown_lines.extend([md_header, md_separator])

    html_lines = [f"""
    <style>
        .perf-table {{ border-collapse: collapse; width: 100%; font-family: sans-serif; margin-bottom: 10px; }}
        .perf-table th, .perf-table td {{ border: 1px solid #ddd; padding: 8px; text-align: center; }}
        .perf-table th {{ background-color: #f2f2f2; }}
        .perf-table .section-header td {{ background-color: #e0e0e0; font-weight: bold; text-align: left; }}
        .perf-table .model-name {{ text-align: left; font-weight: bold; }}
    </style>
    <table class="perf-table">
        <thead>
            <tr>
                <th>Model</th>{''.join(f'<th>{name}</th>' for name in METRICS_MAP.values())}
            </tr>
        </thead>
        <tbody>
    """]

    # --- 3. Populate tables section by section ---
    for source_name, df_data in all_results.items():
        # Add section titles
        section_title_text = f"**{source_name.replace('_', ' ').title()} Data**"
        markdown_lines.append(f"| {section_title_text} |" + " |" * len(METRICS_MAP))
        html_lines.append(f"<tr class='section-header'><td colspan='{len(METRICS_MAP)+1}'>{section_title_text.replace('**','')}</td></tr>")

        df_data = df_data.sort_values(by='model_name')

        for _, model_row in df_data.iterrows():
            model_name = model_row['model_name']
            
            # --- Calculate metrics ---
            final_metrics = {key: model_row.get(key) for key in METRICS_MAP.keys()}
            if 'pr_auc' in final_metrics and (final_metrics['pr_auc'] is None or np.all(pd.isnull(final_metrics['pr_auc']))):
                auprc_scores = []
                base_to_use = model_base_map.get(model_name, DEFAULT_BASE)
                if base_to_use in base_models_data:
                    y_true_runs = base_models_data[base_to_use]
                    y_prob_runs = model_row.get('y_prob')
                    if y_prob_runs is not None:
                        for i in range(len(y_prob_runs)):
                            if i < len(y_true_runs) and len(y_true_runs[i]) == len(y_prob_runs[i]):
                                precision, recall, _ = precision_recall_curve(y_true_runs[i], y_prob_runs[i])
                                auprc_scores.append(auc(recall, precision))
                final_metrics['pr_auc'] = auprc_scores if auprc_scores else None

            # --- Build Markdown Rows ---
            md_mean_values = [model_name]
            md_ci_values = [""]
            
            # --- Build HTML Row ---
            html_row_cells = [f"<td class='model-name'>{model_name}</td>"]

            for metric_key in METRICS_MAP.keys():
                mean_str, ci_str = get_metric_ci(final_metrics.get(metric_key))
                
                md_mean_values.append(mean_str)
                md_ci_values.append(ci_str)
                
                html_cell = f"{mean_str}<br>{ci_str}" if ci_str != "N/A" else mean_str
                html_row_cells.append(f"<td>{html_cell}</td>")

            markdown_lines.append("| " + " | ".join(md_mean_values) + " |")
            markdown_lines.append("| " + " | ".join(md_ci_values) + " |")
            html_lines.append("<tr>" + "".join(html_row_cells) + "</tr>")
    
    html_lines.append("</tbody></table>")
    
    # --- 4. Finalize and Output ---
    # --- Markdown Output ---
    final_markdown = "\n".join(markdown_lines)
    try:
        with open(OUTPUT_FILE, 'w') as f:
            f.write(final_markdown)
        print(f"\nSuccessfully saved Markdown table to: {OUTPUT_FILE}")
    except IOError as e:
        print(f"\nERROR: Could not save Markdown file. {e}")
        
    print("\n--- Generated Markdown Table ---\n")
    print(final_markdown)
    
    # --- HTML Output (for Jupyter) ---
    table_html = "\n".join(html_lines)
    
    # MODIFIED: Create a collapsible widget for the HTML source
    collapsible_id = f"collapsible-{uuid.uuid4().hex}"
    escaped_html = html.escape(table_html)
    
    full_html_output = f"""
    <div>
        {table_html}
        <a href="javascript:void(0);" onclick="document.getElementById('{collapsible_id}').style.display = document.getElementById('{collapsible_id}').style.display === 'none' ? 'block' : 'none';">
            Show/Hide HTML Source
        </a>
        <pre id="{collapsible_id}" style="display:none; background-color:#f8f8f8; border:1px solid #ccc; padding:10px; white-space: pre-wrap; word-wrap: break-word;"><code>{escaped_html}</code></pre>
    </div>
    """
    
    try:
        from IPython.display import display, HTML
        print("\n--- Rich Text Table (for Jupyter Notebook) ---")
        display(HTML(full_html_output))
    except ImportError:
        print("\n--- HTML Output (for non-Jupyter environments) ---")
        print("To see the rich text table, run this script in a Jupyter Notebook or open the saved HTML file.")
        try:
            html_file_name = OUTPUT_FILE.replace('.md', '.html')
            with open(html_file_name, 'w') as f:
                f.write(full_html_output)
            print(f"HTML table saved to: {html_file_name}")
        except IOError as e:
             print(f"\nERROR: Could not save HTML file. {e}")


if __name__ == "__main__":
    generate_performance_tables()


Successfully loaded data for: PREOP
Successfully loaded data for: INTRAOP
Successfully loaded data for: COMBINED

Successfully saved Markdown table to: performance_table.md

--- Generated Markdown Table ---

| Model | AUROC | AUPRC | Sensitivity | Specificity | Precision | F-score | Accuracy |
|---|---|---|---|---|---|---|---|
| **Preop Data** | | | | | | | |
| ASA Rule | 0.758 | 0.192 | 0.067 | 0.992 | 0.295 | 0.109 | 0.530 |
|  | (0.755, 0.761) | (0.185, 0.199) | (0.063, 0.071) | (0.992, 0.993) | (0.280, 0.310) | (0.103, 0.116) | (0.528, 0.532) |
| AutoGluon | 0.933 | 0.654 | 0.454 | 0.996 | 0.830 | 0.587 | 0.725 |
|  | (0.931, 0.935) | (0.650, 0.658) | (0.447, 0.461) | (0.995, 0.996) | (0.822, 0.839) | (0.580, 0.593) | (0.721, 0.728) |
| GBT | 0.926 | 0.614 | 0.626 | 0.972 | 0.512 | 0.564 | 0.799 |
|  | (0.924, 0.928) | (0.609, 0.620) | (0.620, 0.633) | (0.971, 0.973) | (0.506, 0.518) | (0.559, 0.568) | (0.796, 0.802) |
| KNN | 0.909 | 0.542 | 0.126 | 1.000 | 0.949 | 0.223 | 0.563 |